<center> <h2> Intelligent Job Market Recommendation System - EDA & Preprocessing </h2> </center>

### 1. Import Libraries ###

In [34]:
!pip install sentence_transformers
!pip install keybert

In [35]:
# Data Cleaning
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Data Preprocessing
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from keybert import KeyBERT
from sklearn.feature_extraction.text import TfidfVectorizer
import re

### 2. Load Dataset ###

In [2]:
# Load the directory
data_path = os.path.join("postings.csv")

# Load dataset
df = pd.read_csv(data_path)

In [3]:
df.sample(5)

,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
111800,3905397535,California Department of Health Care Services,Eligibility and Benefits Data Section Chief (H...,Job Description And Duties\n\nWhy Join DHCS?\n...,13225.0,MONTHLY,"Sacramento, CA",3753860.0,5.0,NaN,...,NaN,1.713504e+12,calcareers.ca.gov,0,FULL_TIME,USD,BASE_SALARY,143040.0,95811.0,6067.0
53616,3901658458,Park Place Technologies,Test Automation Engineer Lead/Analyst- Atlanta,This role will require 4 days in office at our...,NaN,NaN,"Atlanta, GA",825290.0,91.0,NaN,...,NaN,1.713212e+12,NaN,0,FULL_TIME,NaN,NaN,NaN,30303.0,13121.0
13706,3887838647,CareerStaff Unlimited,Registered Nurse - RN - LTAC,Registered Nurse - RN - LTAC\nCareerStaff Unli...,NaN,NaN,"Tucson, AZ",3706049.0,3.0,NaN,...,NaN,1.712364e+12,www.click2apply.net,0,FULL_TIME,NaN,NaN,NaN,85701.0,4019.0
102577,3905283775,The Chefs'​ Warehouse,Senior Specialty Buyer,About The Chefs' Warehouse\n\nThe Chefs' Wareh...,NaN,NaN,"Ridgefield, CT",107699.0,4.0,NaN,...,NaN,1.713472e+12,us232.dayforcehcm.com,0,FULL_TIME,NaN,NaN,NaN,6877.0,9001.0
10697,3887102727,Sunrise ShopRite,Night Operations Manager,Job description We are living our Purpose – To...,NaN,NaN,"Parsippany, NJ",35682245.0,3.0,NaN,...,NaN,1.712670e+12,NaN,0,FULL_TIME,NaN,NaN,NaN,7054.0,34027.0


### 3. Static Information ###

#### Surface-level Inspection ####

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

In [5]:
df.describe()

,job_id,max_salary,company_id,views,med_salary,min_salary,applies,original_listed_time,remote_allowed,expiry,closed_time,listed_time,sponsored,normalized_salary,zip_code,fips
count,1.238490e+05,2.979300e+04,1.221320e+05,122160.000000,6280.000000,2.979300e+04,23320.000000,1.238490e+05,15246.0,1.238490e+05,1.073000e+03,1.238490e+05,123849.0,3.607300e+04,102977.000000,96434.000000
mean,3.896402e+09,9.193942e+04,1.220401e+07,14.618247,22015.619876,6.491085e+04,10.591981,1.713152e+12,1.0,1.716213e+12,1.712928e+12,1.713204e+12,0.0,2.053270e+05,50400.491887,28713.879887
std,8.404355e+07,7.011101e+05,2.554143e+07,85.903598,52255.873846,4.959738e+05,29.047395,4.848209e+08,0.0,2.321394e+09,3.622893e+08,3.989122e+08,0.0,5.097627e+06,30252.232515,16015.929825
min,9.217160e+05,1.000000e+00,1.009000e+03,1.000000,0.000000,1.000000e+00,1.000000,1.701811e+12,1.0,1.712903e+12,1.712346e+12,1.711317e+12,0.0,0.000000e+00,1001.000000,1003.000000
25%,3.894587e+09,4.828000e+01,1.435200e+04,3.000000,18.940000,3.700000e+01,1.000000,1.712863e+12,1.0,1.715481e+12,1.712670e+12,1.712886e+12,0.0,5.200000e+04,24112.000000,13121.000000
50%,3.901998e+09,8.000000e+04,2.269650e+05,4.000000,25.500000,6.000000e+04,3.000000,1.713395e+12,1.0,1.716042e+12,1.712670e+12,1.713408e+12,0.0,8.150000e+04,48059.000000,29183.000000
75%,3.904707e+09,1.400000e+05,8.047188e+06,8.000000,2510.500000,1.000000e+05,8.000000,1.713478e+12,1.0,1.716088e+12,1.713283e+12,1.713484e+12,0.0,1.250000e+05,78201.000000,42077.000000
max,3.906267e+09,1.200000e+08,1.034730e+08,9975.000000,750000.000000,8.500000e+07,967.000000,1.713573e+12,1.0,1.729125e+12,1.713562e+12,1.713573e+12,0.0,5.356000e+08,99901.000000,56045.000000


#### NaN / Null & Missing Value Detection ####

In [6]:
na_null_summary = pd.DataFrame({
    'df_isna' : df.isna().sum(), # isna().sum() return the count and sum of NaN / Null values
    'df_isnull' : df.isnull().sum(),
})

print(na_null_summary)

                            df_isna  df_isnull
job_id                            0          0
company_name                   1719       1719
title                             0          0
description                       7          7
max_salary                    94056      94056
pay_period                    87776      87776
location                          0          0
company_id                     1717       1717
views                          1689       1689
med_salary                   117569     117569
min_salary                    94056      94056
formatted_work_type               0          0
applies                      100529     100529
original_listed_time              0          0
remote_allowed               108603     108603
job_posting_url                   0          0
application_url               36665      36665
application_type                  0          0
expiry                            0          0
closed_time                  122776     122776
formatted_exp

#### Duplicated Value Detection ####

In [7]:
duplicated_summary = pd.DataFrame({
    'duplicates': [df.duplicated().sum()]
}, index=['df'])

print(duplicated_summary)

    duplicates
df           0


#### Categorical Column Counters ####

In [8]:
df_cols = df.columns

comparison_df = pd.DataFrame({
    'df_columns': df_cols,
})

print("=========== List of all categorical features ===========")
print(comparison_df)

=========== List of all categorical features ===========
                    df_columns
0                       job_id
1                 company_name
2                        title
3                  description
4                   max_salary
5                   pay_period
6                     location
7                   company_id
8                        views
9                   med_salary
10                  min_salary
11         formatted_work_type
12                     applies
13        original_listed_time
14              remote_allowed
15             job_posting_url
16             application_url
17            application_type
18                      expiry
19                 closed_time
20  formatted_experience_level
21                 skills_desc
22                 listed_time
23              posting_domain
24                   sponsored
25                   work_type
26                    currency
27           compensation_type
28           normalized_salary
29           

### 4. Data Cleaning ###

#### Drop Unnecessary Columns ####

In [9]:
interim_df = df.drop(
    columns = ["max_salary", 
               "med_salary", 
               "min_salary", 
               "pay_period", 
               "applies",
              "remote_allowed",
              "closed_time",
              "skills_desc",
              "currency",
              "compensation_type",
              "views",
              "job_posting_url",
              "application_url",
              "zip_code",
              "closed_time",
              "original_listed_time",
              "expiry"]
)

In [10]:
interim_df.describe()

,job_id,company_id,listed_time,sponsored,normalized_salary,fips
count,1.238490e+05,1.221320e+05,1.238490e+05,123849.0,3.607300e+04,96434.000000
mean,3.896402e+09,1.220401e+07,1.713204e+12,0.0,2.053270e+05,28713.879887
std,8.404355e+07,2.554143e+07,3.989122e+08,0.0,5.097627e+06,16015.929825
min,9.217160e+05,1.009000e+03,1.711317e+12,0.0,0.000000e+00,1003.000000
25%,3.894587e+09,1.435200e+04,1.712886e+12,0.0,5.200000e+04,13121.000000
50%,3.901998e+09,2.269650e+05,1.713408e+12,0.0,8.150000e+04,29183.000000
75%,3.904707e+09,8.047188e+06,1.713484e+12,0.0,1.250000e+05,42077.000000
max,3.906267e+09,1.034730e+08,1.713573e+12,0.0,5.356000e+08,56045.000000


In [11]:
interim_df

,job_id,company_name,title,description,location,company_id,formatted_work_type,application_type,formatted_experience_level,listed_time,posting_domain,sponsored,work_type,normalized_salary,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,"Princeton, NJ",2774458.0,Full-time,ComplexOnsiteApply,NaN,1.713398e+12,NaN,0,FULL_TIME,38480.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...","Fort Collins, CO",NaN,Full-time,ComplexOnsiteApply,NaN,1.712858e+12,NaN,0,FULL_TIME,83200.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,"Cincinnati, OH",64896719.0,Full-time,ComplexOnsiteApply,NaN,1.713278e+12,NaN,0,FULL_TIME,55000.0,39061.0
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,"New Hyde Park, NY",766262.0,Full-time,ComplexOnsiteApply,NaN,1.712896e+12,NaN,0,FULL_TIME,157500.0,36059.0
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,"Burlington, IA",NaN,Full-time,ComplexOnsiteApply,NaN,1.713452e+12,NaN,0,FULL_TIME,70000.0,19057.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123844,3906267117,Lozano Smith,Title IX/Investigations Attorney,Our Walnut Creek office is currently seeking a...,"Walnut Creek, CA",56120.0,Full-time,ComplexOnsiteApply,Mid-Senior level,1.713571e+12,NaN,0,FULL_TIME,157500.0,6013.0
123845,3906267126,Pinterest,"Staff Software Engineer, ML Serving Platform",About Pinterest:\n\nMillions of people across ...,United States,1124131.0,Full-time,OffsiteApply,Mid-Senior level,1.713572e+12,www.pinterestcareers.com,0,FULL_TIME,NaN,NaN
123846,3906267131,EPS Learning,"Account Executive, Oregon/Washington",Company Overview\n\nEPS Learning is a leading ...,"Spokane, WA",90552133.0,Full-time,OffsiteApply,Mid-Senior level,1.713572e+12,epsoperations.bamboohr.com,0,FULL_TIME,NaN,53063.0
123847,3906267195,Trelleborg Applied Technologies,Business Development Manager,The Business Development Manager is a 'hunter'...,"Texas, United States",2793699.0,Full-time,ComplexOnsiteApply,NaN,1.713573e+12,NaN,0,FULL_TIME,NaN,NaN


### 5. Data Preprocessing ###

#### Safe Copy ####

In [21]:
cleaned_df = interim_df.copy()

#### Categorical Decomposition ####

In [22]:
cleaned_df[["state", "city"]] = interim_df["location"].str.split(",", n = 1, expand = True)

In [23]:
cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 17 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   location                    123849 non-null  object 
 5   company_id                  122132 non-null  float64
 6   formatted_work_type         123849 non-null  object 
 7   application_type            123849 non-null  object 
 8   formatted_experience_level  94440 non-null   object 
 9   listed_time                 123849 non-null  float64
 10  posting_domain              83881 non-null   object 
 11  sponsored                   123849 non-null  int64  
 12  work_type                   123849 non-null  object 
 13  normalized_sal

#### Timeline Derivation (Use for kNN, Linear, Tree-based models & Neural networks) ####

In [24]:
# Ensure numeric and convert epoch ms → datetime (UTC)
cleaned_df["listed_datetime"] = pd.to_datetime(cleaned_df["listed_time"], unit = "ms", utc = True)

# Posting year
cleaned_df["posting_year"] = cleaned_df["listed_datetime"].dt.year

# Posting month
cleaned_df["posting_month"] = cleaned_df["listed_datetime"].dt.month

# Posting day
cleaned_df["posting_dow"] = cleaned_df["listed_datetime"].dt.weekday

In [25]:
# Encoded posting day / month / year
cleaned_df["posting_month_sin"] = np.sin(2 * np.pi * cleaned_df["listed_datetime"].dt.month / 12)
cleaned_df["posting_month_cos"] = np.cos(2 * np.pi * cleaned_df["listed_datetime"].dt.month / 12)

cleaned_df['posting_dow_sin'] = np.sin(2 * np.pi * cleaned_df['listed_datetime'].dt.weekday / 7)
cleaned_df['posting_dow_cos'] = np.cos(2 * np.pi * cleaned_df['listed_datetime'].dt.weekday / 7)

In [27]:
# Posting recency (numerical)
now = pd.Timestamp.utcnow()

cleaned_df["posting_recency_days"] = (now - cleaned_df["listed_datetime"]).dt.days

# Drop raw timestamp columns
cleaned_df.drop(columns = ['listed_time', 'listed_datetime'], inplace = True)

In [30]:
cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 24 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   location                    123849 non-null  object 
 5   company_id                  122132 non-null  float64
 6   formatted_work_type         123849 non-null  object 
 7   application_type            123849 non-null  object 
 8   formatted_experience_level  94440 non-null   object 
 9   posting_domain              83881 non-null   object 
 10  sponsored                   123849 non-null  int64  
 11  work_type                   123849 non-null  object 
 12  normalized_salary           36073 non-null   float64
 13  fips          

#### Task-specific Preprocessing Pipeline ####

#### A. Job Similarity (FAISS, Annoy, HNSW, Cosine Similarity) ####

#### 1. Text Preprocessing → Embeddings ####

In [ ]:
def clean_text(s):
    if pd.isna(s):
        return ""
    s = s.lower()
    s = re.sub(r"\s+", " ", s)
    return s.strip()

cleaned_df["title_clean"] = cleaned_df["title"].apply(clean_text)
cleaned_df["description_clean"] = cleaned_df["description"].apply(clean_text)

In [ ]:
# Sentence-BERT embeddings (frozen)
model = SentenceTransformer("all-MiniLM-L6-v2")

title_emb = model.encode(cleaned_df["title_clean"].tolist(), batch_size = 64, show_progress_bar = True, normalize_embeddings = False)
desc_emb = model.encode(cleaned_df["description_clean"].tolist(), batch_size = 32, show_progress_bar = True, normalize_embeddings = False)

#### 2. Categorical (low-noise only) ####

In [ ]:
# formatted_work_type → One-Hot
ohe_work = OneHotEncoder(sparse=False, handle_unknown="ignore")

work_type_ohe = ohe_work.fit_transform(cleaned_df[['formatted_work_type']])

# formatted_experience_level → Ordinal
exp_order = ['Internship', 'Entry level', 'Associate', 'Mid-Senior level', 'Director', 'Executive']

ord_exp = OrdinalEncoder(categories = [exp_order], handle_unknown = 'use_encoded_value', unknown_value = -1)

exp_ord = ord_exp.fit_transform(cleaned_df[['formatted_experience_level']])

# sponsored → binary 
sponsored_bin = cleaned_df[['sponsored']].values

#### 3. Numerical Features ####

In [ ]:
# normalized_salary → RobustScaler
salary_scaler = RobustScaler()

salary_scaled = salary_scaler.fit_transform(cleaned_df[['normalized_salary']].fillna(cleaned_df['normalized_salary'].median()))

# posting_recency_days → MinMaxScaler
recency_scaler = MinMaxScaler()

recency_scaled = recency_scaler.fit_transform(cleaned_df[['posting_recency_days']])

# fips
fips_raw = cleaned_df[['fips']].fillna(-1).values

#### 4. Final Similarity Vector (Concatenation) ####

In [ ]:
job_similarity_matrix = np.hstack([
    title_emb * 1.5,        # dominant
    desc_emb * 2.0,         # dominant
    salary_scaled,
    recency_scaled,
    sponsored_bin,
    exp_ord,
    work_type_ohe
])

#### B. Salary Prediction (Regression - XGBoost, LightGBM, CatBoost) ####

In [ ]:
# Target
y = cleaned_df["normalized_salary"]

In [ ]:
# Numerical Preprocessing
# posting_recency_days → StandardScaler
recency_std = StandardScaler()
X_recency = recency_std.fit_transform(cleaned_df[['posting_recency_days']])

# fips → Target Mean Encoding (county salary)
fips_salary_mean = (cleaned_df.groupby('fips')['normalized_salary'].mean())

X_fips = cleaned_df['fips'].map(fips_salary_mean)
X_fips = X_fips.fillna(fips_salary_mean.mean()).values.reshape(-1, 1)

# title → frozen embeddings (reuse)
X_title_emb = title_emb

# formatted_work_type → One-Hot (reuse)
X_work_type = work_type_ohe

# experience_level → Ordinal (reuse)
X_exp = exp_ord

# Final regression matrix
X_salary = np.hstack([X_title_emb, X_company, X_fips, X_recency, X_exp, X_work_type])

#### C. Skill-Based Recommendations ####

#### 1. Keyphrase Extraction ####

In [ ]:
kw_model = KeyBERT(model)

def extract_skills(text):
    keywords = kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words='english', top_n=10)
    return [k[0] for k in keywords]

cleaned_df['skills'] = (cleaned_df['title_clean'] + " " + cleaned_df['description_clean']).apply(extract_skills)

#### 2. Job → Skill TF-IDF ####

In [ ]:
skill_text = cleaned_df['skills'].apply(lambda x: " ".join(x))

tfidf = TfidfVectorizer(min_df=5)
job_skill_matrix = tfidf.fit_transform(skill_text)

# 📌 No scaling
# 📌 Cosine similarity only

#### D. Market Analytics (Aggregation-First) ####

In [ ]:
# Salary
cleaned_df['log_salary'] = np.log1p(cleaned_df['normalized_salary'])

# Posting counts (time-based)
cleaned_df['week'] = (pd.to_datetime(cleaned_df['posting_year'], format='%Y'))

weekly_postings = (cleaned_df.groupby(['fips', 'posting_year', 'posting_month']).size().reset_index(name='posting_count'))

# 📌 No StandardScaler
# 📌 Aggregation > modeling